# cyp51A Gene Variants Analysis - Iteration 1

This notebook identifies and visualizes variants falling within the cyp51A gene (Afu4g06890) using:
- GTF dataset #4 for gene structure
- VCF file collections from history #450 and #351 for variant data
- Creates heatmap showing variants in coding regions with local coordinates

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.patches import Rectangle
import re
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# Set up plotting parameters
plt.style.use('default')
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 10

## Step 1: Load GTF Data and Find cyp51A Gene

In [ ]:
# Load GTF file (dataset #4)
gtf_file = 'dataset_4.dat'  # GTF annotation file from Galaxy history
gtf_columns = ['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute']

print("Loading GTF data...")
try:
    gtf_data = pd.read_csv(gtf_file, sep='\t', comment='#', names=gtf_columns, dtype=str)
    print(f"✓ Loaded GTF file with {len(gtf_data)} entries")
except Exception as e:
    print(f"✗ Error loading GTF file: {e}")
    print("Available files in current directory:")
    print([f for f in os.listdir('.') if f.startswith('dataset')][:10])  # Show first 10 dataset files

In [ ]:
# Find cyp51A gene (Afu4g06890)
def extract_gene_info(attribute_string):
    """Extract gene ID and other info from GTF attributes"""
    if pd.isna(attribute_string):
        return None, None
    
    gene_id = None
    gene_name = None
    
    # Look for gene_id, ID, Name, locus_tag
    patterns = [
        (r'gene_id\s*["=]([^\s;";]+)', 'gene_id'),
        (r'ID=([^\s;]+)', 'ID'),
        (r'Name=([^\s;]+)', 'Name'),
        (r'locus_tag\s*["=]([^\s;";]+)', 'locus_tag')
    ]
    
    for pattern, field in patterns:
        match = re.search(pattern, str(attribute_string), re.IGNORECASE)
        if match:
            if field in ['gene_id', 'ID', 'locus_tag']:
                gene_id = match.group(1)
            elif field == 'Name':
                gene_name = match.group(1)
    
    return gene_id, gene_name

if 'gtf_data' in locals():
    # Extract gene info
    gene_info = gtf_data['attribute'].apply(extract_gene_info)
    gtf_data['gene_id'] = [info[0] for info in gene_info]
    gtf_data['gene_name'] = [info[1] for info in gene_info]
    
    # Search for cyp51A gene (Afu4g06890)
    cyp51a_mask = (
        gtf_data['attribute'].str.contains('Afu4g06890', case=False, na=False) |
        gtf_data['gene_id'].str.contains('Afu4g06890', case=False, na=False) |
        gtf_data['attribute'].str.contains('cyp51A', case=False, na=False)
    )
    
    cyp51a_entries = gtf_data[cyp51a_mask].copy()
    
    print(f"\nFound {len(cyp51a_entries)} entries for cyp51A (Afu4g06890)")
    
    if len(cyp51a_entries) > 0:
        print("\ncyp51A gene entries:")
        display_cols = ['seqname', 'feature', 'start', 'end', 'strand']
        print(cyp51a_entries[display_cols].head(10))
        
        # Get basic gene info
        chromosome = cyp51a_entries.iloc[0]['seqname']
        strand = cyp51a_entries.iloc[0]['strand']
        
        # Convert coordinates to int
        cyp51a_entries['start'] = cyp51a_entries['start'].astype(int)
        cyp51a_entries['end'] = cyp51a_entries['end'].astype(int)
        
        gene_start = cyp51a_entries['start'].min()
        gene_end = cyp51a_entries['end'].max()
        
        print(f"\ncyp51A gene location: {chromosome}:{gene_start:,}-{gene_end:,} ({strand} strand)")
        
    else:
        print("\n⚠️ cyp51A gene not found. Searching for similar entries...")
        similar = gtf_data[gtf_data['attribute'].str.contains('cyp|CYP', case=False, na=False)]
        print(f"Found {len(similar)} entries containing 'cyp'")
        if len(similar) > 0:
            print(similar[['seqname', 'feature', 'attribute']].head())
else:
    print("GTF data not available")

## Step 2: Extract CDS Regions and Create Local Coordinate Mapping

In [ ]:
if 'cyp51a_entries' in locals() and len(cyp51a_entries) > 0:
    # Extract CDS entries
    cds_entries = cyp51a_entries[cyp51a_entries['feature'] == 'CDS'].copy()
    
    if len(cds_entries) == 0:
        # Try exon or other features if no CDS
        cds_entries = cyp51a_entries[cyp51a_entries['feature'].isin(['exon', 'mRNA', 'transcript'])].copy()
        print(f"No CDS entries found, using {len(cds_entries)} {cds_entries['feature'].iloc[0] if len(cds_entries) > 0 else 'N/A'} entries")
    else:
        print(f"Found {len(cds_entries)} CDS entries")
    
    if len(cds_entries) > 0:
        # Get CDS coordinates
        cds_coords = []
        for _, cds in cds_entries.iterrows():
            start, end = int(cds['start']), int(cds['end'])
            cds_coords.append((start, end))
            print(f"  CDS: {start:,} - {end:,} ({end-start+1} bp)")
        
        # Sort CDS coordinates
        cds_coords.sort()
        
        # Create local coordinate mapping (1-based, coding sequence only)
        local_coords = {}  # genomic_pos -> local_pos
        genomic_to_local = {}  # for quick lookup
        local_pos = 1
        
        print(f"\nCreating local coordinate mapping:")
        for i, (start, end) in enumerate(cds_coords, 1):
            cds_length = end - start + 1
            print(f"  Exon {i}: genomic {start:,}-{end:,} → local {local_pos}-{local_pos + cds_length - 1}")
            
            for genomic_pos in range(start, end + 1):
                local_coords[genomic_pos] = local_pos
                genomic_to_local[local_pos] = genomic_pos
                local_pos += 1
        
        total_coding_length = local_pos - 1
        print(f"\nTotal coding sequence: {total_coding_length} nucleotides")
        
        # Define region of interest for variant analysis
        roi_start = gene_start - 1000  # 1kb upstream
        roi_end = gene_end + 1000      # 1kb downstream
        
        print(f"Region of interest: {chromosome}:{roi_start:,}-{roi_end:,}")
        
    else:
        print("No CDS or exon entries found for cyp51A")
        cds_coords = []
        local_coords = {}
        total_coding_length = 0

else:
    print("cyp51A gene data not available")
    cds_coords = []
    local_coords = {}
    total_coding_length = 0

## Step 3: Load and Process VCF Files from Collections #450 and #351

In [ ]:
# Find VCF files from history collections #450 and #351
print("Searching for VCF files from collections #450 and #351...")

# Look for VCF files in current directory
vcf_files = []
all_files = os.listdir('.')

# Look for dataset files that might be VCF
potential_vcf = [f for f in all_files if 'dataset' in f and any(x in f.lower() for x in ['vcf', '450', '351'])]
print(f"Potential VCF files found: {potential_vcf[:10]}...")  # Show first 10

# Try to identify VCF files by checking file content
vcf_data = {}
sample_variants = {}

# Check specific dataset files that might be from collections 450 and 351
target_datasets = [f'dataset_{i}.dat' for i in [450, 351]] + \
                 [f for f in all_files if any(x in f for x in ['450', '351']) and 'dataset' in f]

print(f"Checking target datasets: {target_datasets}")

def load_vcf_file(filename):
    """Try to load a VCF file and extract variants"""
    try:
        # Try reading as VCF
        with open(filename, 'r') as f:
            lines = f.readlines()[:10]  # Check first 10 lines
            
        # Check if it looks like VCF
        has_vcf_header = any(line.startswith('##fileformat=VCF') for line in lines)
        has_chrom_line = any(line.startswith('#CHROM') for line in lines)
        
        if has_vcf_header or has_chrom_line:
            print(f"  ✓ {filename} appears to be VCF format")
            
            # Read VCF data
            vcf_df = pd.read_csv(filename, sep='\t', comment='#', 
                               names=['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO', 'FORMAT'] + 
                                     [f'SAMPLE_{i}' for i in range(20)],  # Allow for multiple samples
                               dtype=str, na_values='.', 
                               error_bad_lines=False, warn_bad_lines=False)
            
            # Filter for cyp51A region if coordinates are available
            if len(vcf_df) > 0 and 'roi_start' in locals():
                vcf_df['POS'] = pd.to_numeric(vcf_df['POS'], errors='coerce')
                
                # Filter for region of interest
                region_variants = vcf_df[
                    (vcf_df['CHROM'] == chromosome) & 
                    (vcf_df['POS'] >= roi_start) & 
                    (vcf_df['POS'] <= roi_end)
                ].copy()
                
                if len(region_variants) > 0:
                    print(f"    Found {len(region_variants)} variants in cyp51A region")
                    return region_variants
                else:
                    print(f"    No variants found in cyp51A region")
                    return pd.DataFrame()
            else:
                print(f"    VCF has {len(vcf_df)} total variants")
                return vcf_df.head(100)  # Return first 100 for analysis
        else:
            return None
            
    except Exception as e:
        print(f"  ✗ Error reading {filename}: {str(e)[:100]}...")
        return None

# Try to load VCF files
vcf_datasets = {}
for filename in target_datasets[:5]:  # Check first 5 potential files
    if os.path.exists(filename):
        print(f"\nChecking {filename}...")
        vcf_result = load_vcf_file(filename)
        if vcf_result is not None and len(vcf_result) > 0:
            vcf_datasets[filename] = vcf_result

print(f"\nLoaded {len(vcf_datasets)} VCF datasets with variant data")

## Step 4: Analyze Variants in cyp51A Coding Regions

In [ ]:
# Process variant data for coding regions
coding_variants = []
variant_matrix = None

if len(vcf_datasets) > 0 and len(local_coords) > 0:
    print("Analyzing variants in coding regions...")
    
    # Combine all VCF data
    all_variants = pd.DataFrame()
    for filename, vcf_data in vcf_datasets.items():
        vcf_data['source_file'] = filename
        all_variants = pd.concat([all_variants, vcf_data], ignore_index=True)
    
    if len(all_variants) > 0:
        print(f"Total variants loaded: {len(all_variants)}")
        
        # Filter variants that fall in coding regions
        all_variants['POS'] = pd.to_numeric(all_variants['POS'], errors='coerce')
        coding_variants_df = all_variants[
            all_variants['POS'].isin(local_coords.keys())
        ].copy()
        
        if len(coding_variants_df) > 0:
            # Add local coordinates
            coding_variants_df['local_pos'] = coding_variants_df['POS'].map(local_coords)
            
            print(f"\nFound {len(coding_variants_df)} variants in coding regions:")
            print(coding_variants_df[['CHROM', 'POS', 'local_pos', 'REF', 'ALT', 'source_file']].head())
            
            # Create variant matrix for heatmap
            # Rows: genomic positions with variants
            # Columns: different samples or variant types
            
            variant_positions = sorted(coding_variants_df['local_pos'].unique())
            variant_info = []
            
            for local_pos in variant_positions:
                genomic_pos = next(gpos for gpos, lpos in local_coords.items() if lpos == local_pos)
                pos_variants = coding_variants_df[coding_variants_df['local_pos'] == local_pos]
                
                variant_info.append({
                    'local_position': local_pos,
                    'genomic_position': genomic_pos,
                    'num_variants': len(pos_variants),
                    'ref_alleles': ','.join(pos_variants['REF'].unique()),
                    'alt_alleles': ','.join(pos_variants['ALT'].unique()),
                    'variant_types': ','.join(pos_variants['source_file'].unique())
                })
            
            variant_summary = pd.DataFrame(variant_info)
            print(f"\nVariant summary:")
            print(variant_summary.head())
            
        else:
            print("No variants found in coding regions")
            variant_summary = pd.DataFrame()
    else:
        print("No variant data available")
        variant_summary = pd.DataFrame()
else:
    print("Cannot analyze variants - missing VCF data or gene coordinates")
    variant_summary = pd.DataFrame()

## Step 5: Create Heatmap Visualization

In [ ]:
# Create comprehensive heatmap showing genomic neighborhood and variants
if len(local_coords) > 0:
    print("Creating heatmap visualization...")
    
    # Create figure with subplots
    fig, axes = plt.subplots(3, 1, figsize=(20, 12), height_ratios=[1, 2, 4])
    
    # Panel 1: Gene structure overview
    ax1 = axes[0]
    ax1.set_xlim(roi_start, roi_end)
    ax1.set_ylim(-1, 2)
    
    # Draw gene body
    gene_rect = Rectangle((gene_start, 0), gene_end - gene_start, 0.5, 
                         facecolor='lightblue', edgecolor='blue', alpha=0.7)
    ax1.add_patch(gene_rect)
    
    # Draw CDS regions
    for start, end in cds_coords:
        cds_rect = Rectangle((start, 0), end - start, 0.5, 
                            facecolor='red', edgecolor='darkred', alpha=0.8)
        ax1.add_patch(cds_rect)
    
    # Mark variants if available
    if 'variant_summary' in locals() and len(variant_summary) > 0:
        for _, variant in variant_summary.iterrows():
            ax1.axvline(x=variant['genomic_position'], color='orange', linewidth=2, alpha=0.8)
    
    ax1.set_title(f'cyp51A Gene (Afu4g06890) - Genomic Context\n{chromosome}:{roi_start:,}-{roi_end:,}', 
                  fontsize=14, fontweight='bold')
    ax1.set_ylabel('Gene\nStructure')
    ax1.set_xticks([])
    
    # Panel 2: Variant density along gene
    ax2 = axes[1]
    
    if 'variant_summary' in locals() and len(variant_summary) > 0:
        # Create variant density plot
        ax2.bar(variant_summary['genomic_position'], variant_summary['num_variants'], 
               width=50, color='orange', alpha=0.7, edgecolor='darkorange')
        ax2.set_ylabel('Number of\nVariants')
        ax2.set_title('Variant Density in cyp51A Gene')
    else:
        ax2.text(0.5, 0.5, 'No variants found in coding regions', 
                ha='center', va='center', transform=ax2.transAxes, fontsize=12)
    
    ax2.set_xlim(roi_start, roi_end)
    ax2.set_xticks([])
    
    # Panel 3: Coding region heatmap with local coordinates
    ax3 = axes[2]
    
    if total_coding_length > 0:
        # Create matrix for heatmap
        matrix_rows = 6  # Different data tracks
        matrix = np.zeros((matrix_rows, total_coding_length))
        
        # Create arrays for genomic positions and local positions
        genomic_positions = []
        local_positions = []
        
        # Fill matrix with data
        for local_pos in range(1, total_coding_length + 1):
            genomic_pos = genomic_to_local.get(local_pos, 0)
            genomic_positions.append(genomic_pos)
            local_positions.append(local_pos)
            
            col_idx = local_pos - 1
            
            # Row 0: Local coordinate (normalized)
            matrix[0, col_idx] = local_pos / total_coding_length
            
            # Row 1: Codon position (1, 2, or 3)
            matrix[1, col_idx] = ((local_pos - 1) % 3) + 1
            
            # Row 2: Distance from start (normalized)
            matrix[2, col_idx] = local_pos
            
            # Row 3: Exon number
            exon_num = 1
            for i, (start, end) in enumerate(cds_coords, 1):
                if start <= genomic_pos <= end:
                    exon_num = i
                    break
            matrix[3, col_idx] = exon_num
            
            # Row 4: Variant presence (if variant data available)
            if 'variant_summary' in locals() and len(variant_summary) > 0:
                has_variant = local_pos in variant_summary['local_position'].values
                matrix[4, col_idx] = 1 if has_variant else 0
            else:
                matrix[4, col_idx] = 0
            
            # Row 5: Position within gene (normalized)
            if genomic_pos > 0:
                matrix[5, col_idx] = (genomic_pos - gene_start) / (gene_end - gene_start)
            else:
                matrix[5, col_idx] = 0
        
        # Create heatmap
        im = ax3.imshow(matrix, aspect='auto', cmap='viridis', interpolation='nearest')
        
        # Set labels
        row_labels = ['Local Coordinate', 'Codon Position', 'Distance from Start', 
                     'Exon Number', 'Variant Presence', 'Gene Position']
        ax3.set_yticks(range(matrix_rows))
        ax3.set_yticklabels(row_labels)
        
        # X-axis: local coordinates
        n_ticks = min(20, total_coding_length // 25)
        if n_ticks > 1:
            tick_positions = np.linspace(0, total_coding_length-1, n_ticks, dtype=int)
            tick_labels = [str(pos+1) for pos in tick_positions]
            ax3.set_xticks(tick_positions)
            ax3.set_xticklabels(tick_labels, rotation=45)
        
        ax3.set_xlabel('Local Coding Coordinate (nucleotide position)')
        ax3.set_title(f'Nucleotide-Level Analysis of cyp51A Coding Sequence ({total_coding_length} bp)')
        
        # Add colorbar
        cbar = plt.colorbar(im, ax=ax3, shrink=0.6)
        cbar.set_label('Normalized Value', rotation=270, labelpad=20)
        
    else:
        ax3.text(0.5, 0.5, 'No coding sequence data available', 
                ha='center', va='center', transform=ax3.transAxes, fontsize=16)
    
    # Set common x-axis for top panels
    if 'roi_start' in locals():
        for ax in [ax1, ax2]:
            ax.set_xlim(roi_start, roi_end)
    
    plt.tight_layout()
    plt.savefig('cyp51A_variants_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n=== Analysis Summary ===")
    print(f"Gene: cyp51A (Afu4g06890)")
    if 'chromosome' in locals():
        print(f"Location: {chromosome}:{gene_start:,}-{gene_end:,}")
        print(f"Coding sequence: {total_coding_length} nucleotides")
        print(f"Number of exons: {len(cds_coords)}")
    
    if 'variant_summary' in locals() and len(variant_summary) > 0:
        print(f"Variants in coding region: {len(variant_summary)}")
        print(f"VCF datasets analyzed: {len(vcf_datasets)}")
    else:
        print("No variants found in coding regions")
        
else:
    print("Cannot create heatmap - gene structure data not available")

## Step 6: Export Results

In [ ]:
# Export analysis results
print("Exporting results...")

# 1. Gene structure and coordinate mapping
if len(local_coords) > 0:
    coord_mapping = pd.DataFrame([
        {
            'local_position': local_pos,
            'genomic_position': genomic_pos,
            'chromosome': chromosome,
            'codon_position': ((local_pos - 1) % 3) + 1,
            'exon_number': next(i for i, (s, e) in enumerate(cds_coords, 1) if s <= genomic_pos <= e),
            'gene_id': 'Afu4g06890',
            'gene_name': 'cyp51A'
        }
        for genomic_pos, local_pos in local_coords.items()
    ]).sort_values('local_position')
    
    coord_mapping.to_csv('cyp51A_coordinate_mapping.csv', index=False)
    print(f"✓ Saved coordinate mapping: {len(coord_mapping)} positions")

# 2. Variant summary
if 'variant_summary' in locals() and len(variant_summary) > 0:
    variant_summary.to_csv('cyp51A_variants_summary.csv', index=False)
    print(f"✓ Saved variant summary: {len(variant_summary)} variant positions")
    
    # Detailed variant information
    if 'coding_variants_df' in locals() and len(coding_variants_df) > 0:
        coding_variants_df.to_csv('cyp51A_coding_variants_detailed.csv', index=False)
        print(f"✓ Saved detailed variants: {len(coding_variants_df)} variant calls")

# 3. Gene structure
if len(cds_coords) > 0:
    gene_structure = pd.DataFrame([
        {
            'exon_number': i,
            'genomic_start': start,
            'genomic_end': end,
            'length_bp': end - start + 1,
            'local_start': min(local_coords[pos] for pos in range(start, end + 1) if pos in local_coords),
            'local_end': max(local_coords[pos] for pos in range(start, end + 1) if pos in local_coords)
        }
        for i, (start, end) in enumerate(cds_coords, 1)
    ])
    
    gene_structure.to_csv('cyp51A_gene_structure.csv', index=False)
    print(f"✓ Saved gene structure: {len(gene_structure)} exons")

# 4. Analysis summary
summary_info = {
    'gene_id': 'Afu4g06890',
    'gene_name': 'cyp51A',
    'chromosome': chromosome if 'chromosome' in locals() else 'N/A',
    'gene_start': gene_start if 'gene_start' in locals() else 0,
    'gene_end': gene_end if 'gene_end' in locals() else 0,
    'gene_length': gene_end - gene_start + 1 if 'gene_end' in locals() else 0,
    'coding_length': total_coding_length,
    'num_exons': len(cds_coords),
    'num_vcf_files': len(vcf_datasets),
    'variants_in_coding': len(variant_summary) if 'variant_summary' in locals() else 0,
    'total_variant_calls': len(coding_variants_df) if 'coding_variants_df' in locals() else 0
}

summary_df = pd.DataFrame([summary_info])
summary_df.to_csv('cyp51A_analysis_summary.csv', index=False)
print(f"✓ Saved analysis summary")

print("\n=== Export Complete ===")
print("Files created:")
for filename in ['cyp51A_coordinate_mapping.csv', 'cyp51A_variants_summary.csv', 
                 'cyp51A_coding_variants_detailed.csv', 'cyp51A_gene_structure.csv', 
                 'cyp51A_analysis_summary.csv', 'cyp51A_variants_heatmap.png']:
    if os.path.exists(filename):
        print(f"  ✓ {filename}")